In [132]:
import pandas as pd
import requests
import zipfile
import io
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer

In [133]:
url = "http://files.grouplens.org/datasets/movielens/ml-100k.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

In [134]:
cols = ['movieId','title','release_date','video_release_date','IMDb_URL',
'unknown','Action','Adventure','Animation','Children','Comedy','Crime',
'Documentary','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery',
'Romance','Sci-Fi','Thriller','War','Western']

In [153]:
movies_df = pd.read_csv(z.open('ml-100k/u.item'), sep='|', header=None, names=cols, encoding='latin-1')

In [154]:
def combine_all_genres(row):
    return '|'.join([genre for genre in genre_cols if row[genre]==1])

In [155]:
movies_df['genres'] = movies_df.apply(combine_all_genres, axis=1)

In [156]:
movies = movies_df[['movieId','title','genres']]

In [157]:
movies

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children|Comedy
1,2,GoldenEye (1995),Action|Adventure|Thriller
2,3,Four Rooms (1995),Thriller
3,4,Get Shorty (1995),Action|Comedy|Drama
4,5,Copycat (1995),Crime|Drama|Thriller
...,...,...,...
1677,1678,Mat' i syn (1997),Drama
1678,1679,B. Monkey (1998),Romance|Thriller
1679,1680,Sliding Doors (1998),Drama|Romance
1680,1681,You So Crazy (1994),Comedy


In [158]:
ratings_cols = ['userId', 'movieId', 'rating', 'timestamp']
ratings = pd.read_csv(z.open('ml-100k/u.data'),sep='\t',header=None,names=ratings_cols)
ratings

,userId,movieId,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


In [159]:
movies=movies.copy()
movies['year']=movies.title.str.extract(r'(\d{4})',expand=False)
movies['title']=movies.title.str.replace(r'\(\d{4}\)','',regex=True)
movies['title']=movies['title'].apply(lambda x:x.strip())
movies['genres']=movies['genres'].str.split('|')


In [160]:
movies

,movieId,title,genres,year
0,1,Toy Story,"[Animation, Children, Comedy]",1995
1,2,GoldenEye,"[Action, Adventure, Thriller]",1995
2,3,Four Rooms,[Thriller],1995
3,4,Get Shorty,"[Action, Comedy, Drama]",1995
4,5,Copycat,"[Crime, Drama, Thriller]",1995
...,...,...,...,...
1677,1678,Mat' i syn,[Drama],1997
1678,1679,B. Monkey,"[Romance, Thriller]",1998
1679,1680,Sliding Doors,"[Drama, Romance]",1998
1680,1681,You So Crazy,[Comedy],1994


In [161]:
for index, row in movies.iterrows():
    for gener in row['genres']:
        movies.at[index,gener]=1
movies=movies.fillna(0)
movies.head()

,movieId,title,genres,year,Animation,Children,Comedy,Action,Adventure,Thriller,...,War,Romance,Horror,Musical,Documentary,Western,Fantasy,Film-Noir,Mystery,unknown
0,1,Toy Story,"[Animation, Children, Comedy]",1995,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,GoldenEye,"[Action, Adventure, Thriller]",1995,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,Four Rooms,[Thriller],1995,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,Get Shorty,"[Action, Comedy, Drama]",1995,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,Copycat,"[Crime, Drama, Thriller]",1995,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [162]:
user_movie_matrix = ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)
user_movie_matrix 

movieId,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
userId,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
940,0.0,0.0,0.0,2.0,0.0,0.0,4.0,5.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
941,5.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [204]:
cf_similarity = cosine_similarity(user_movie_matrix.T)
cf_similarity_df = pd.DataFrame(cf_similarity, index=user_movie_matrix.columns, columns=user_movie_matrix.columns)
cf_similarity_df

movieId,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.402382,0.330245,0.454938,0.286714,0.116344,0.620979,0.481114,0.496288,0.273935,...,0.035387,0.0,0.000000,0.000000,0.035387,0.0,0.0,0.0,0.047183,0.047183
2,0.402382,1.000000,0.273069,0.502571,0.318836,0.083563,0.383403,0.337002,0.255252,0.171082,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.078299,0.078299
3,0.330245,0.273069,1.000000,0.324866,0.212957,0.106722,0.372921,0.200794,0.273669,0.158104,...,0.000000,0.0,0.000000,0.000000,0.032292,0.0,0.0,0.0,0.000000,0.096875
4,0.454938,0.502571,0.324866,1.000000,0.334239,0.090308,0.489283,0.490236,0.419044,0.252561,...,0.000000,0.0,0.094022,0.094022,0.037609,0.0,0.0,0.0,0.056413,0.075218
5,0.286714,0.318836,0.212957,0.334239,1.000000,0.037299,0.334769,0.259161,0.272448,0.055453,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.094211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1678,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,1.0,1.0,1.0,0.000000,0.000000
1679,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,1.0,1.0,1.0,0.000000,0.000000
1680,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,1.0,1.0,1.0,0.000000,0.000000


In [205]:
genre_cols=cols[5:] # baraye check kardan in ke hame soton haye gener dar data frame mojod bashad
for g in genre_cols:
    if g not in movies.columns:
        movies[g]=0
    

cb_matrix=movies[genre_cols]
cb_similarity = cosine_similarity(cb_matrix)
cb_similarity_df = pd.DataFrame(cb_similarity, index=movies['movieId'], columns=movies['movieId'])
cb_similarity_df

movieId,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.000000,0.000000,0.333333,0.000000,0.000000,0.000000,0.666667,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.57735,0.000000
2,0.000000,1.000000,0.577350,0.333333,0.333333,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.816497,0.000000,0.000000,0.000000,0.000000,0.000000,0.408248,0.000000,0.00000,0.000000
3,0.000000,0.577350,1.000000,0.000000,0.577350,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.707107,0.000000,0.000000,0.000000,0.000000,0.000000,0.707107,0.000000,0.00000,0.000000
4,0.333333,0.333333,0.000000,1.000000,0.333333,0.577350,0.408248,0.666667,0.577350,0.408248,...,0.408248,0.577350,0.577350,0.577350,0.577350,0.577350,0.000000,0.408248,0.57735,0.577350
5,0.000000,0.333333,0.577350,0.333333,1.000000,0.577350,0.408248,0.333333,0.577350,0.408248,...,0.408248,0.577350,0.577350,0.577350,0.577350,0.577350,0.408248,0.408248,0.00000,0.577350
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1678,0.000000,0.000000,0.000000,0.577350,0.577350,1.000000,0.707107,0.577350,1.000000,0.707107,...,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.707107,0.00000,1.000000
1679,0.000000,0.408248,0.707107,0.000000,0.408248,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.00000,0.000000
1680,0.000000,0.000000,0.000000,0.408248,0.408248,0.707107,0.500000,0.408248,0.707107,0.500000,...,0.000000,0.707107,0.707107,0.707107,0.707107,0.707107,0.500000,1.000000,0.00000,0.707107


In [203]:
def recommend_movies(movie_title, movies_df, cf_df, cb_df,ratings_df ,alpha=0.7, top_n=5):
   matches = movies_df[movies_df['title'].str.contains(movie_title, case=False)]
   if matches.empty:
       print("not found")
       return pd.DataFrame()
   movie_id = matches.movieId.values[0]
    
   combined_score = alpha*cf_df[movie_id] + (1-alpha)*cb_df[movie_id]
   combined_score = combined_score.drop(movie_id) 

   top_movies = combined_score.sort_values(ascending=False).head(top_n)
   recommendations = movies_df[movies_df['movieId'].isin(top_movies.index)][['title','genres']]

   users_who_like_movie = ratings_df[(ratings_df.movieId == movie_id) & (ratings_df.rating >= 4)]['userId'].unique()
   hit_probs = []
   for r_id in top_movies.index:
       users_who_like_r = ratings_df[(ratings_df.movieId == r_id) & (ratings_df.rating >= 4)]['userId'].unique()
       prob = len(set(users_who_like_movie) & set(users_who_like_r)) / len(users_who_like_movie)
       hit_probs.append(prob)

   recommendations['hit_probability'] = hit_probs
   return recommendations
#cf shebahat bar asas raftar karbaran
#cb shebahat bar asa gener
#ehtemal karbari ke film vorodi va film pishnahadi dide/karbari ke film vorodi ro dide

In [202]:
recommend_movies("Toy Story", movies, cf_similarity_df, cb_similarity_df,ratings,alpha=0.7,top_n=5)

,title,genres,hit_probability
24,"Birdcage, The",[Comedy],0.428571
70,"Lion King, The","[Animation, Children, Musical]",0.394958
94,Aladdin,"[Animation, Children, Comedy, Musical]",0.378151
150,Willy Wonka and the Chocolate Factory,"[Adventure, Children, Comedy]",0.361345
587,Beauty and the Beast,"[Animation, Children, Musical]",0.285714
